# Docling as a Tool for AI Agents via MCP
*Exposing document processing capabilities through the Model Context Protocol*

## Lab 6: Docling + MCP for AI Agents

In this lab, you'll learn how to expose Docling's document processing capabilities to AI agents using the **Model Context Protocol (MCP)**. This enables:

- **AI-Native Integration**: LLMs can directly invoke Docling tools during conversations
- **Agentic Document Processing**: AI agents can convert, create, and export documents autonomously
- **Fully Open-Source Stack**: docling-mcp + Continue + Granite via Ollama
- **Zero Custom Code**: Use the official docling-mcp package — no MCP server coding required

### What is MCP?

The [Model Context Protocol (MCP)](https://modelcontextprotocol.io/) is an open standard that allows AI models to discover and use external tools. Instead of calling APIs directly, the LLM can invoke tools through a standardized protocol, making AI agents more capable and extensible.

### How This Connects to Lab 4

In Lab 4, you deployed Docling as a REST API (docling-serve) for programmatic access. MCP takes this further: instead of *you* writing code to call the API, the *AI agent* discovers and calls Docling tools autonomously. This is the modern pattern for integrating capabilities into AI workflows.

## Prerequisites

Before we begin, ensure you have:
- Python 3.10 or later installed
- [VS Code](https://code.visualstudio.com/) installed
- [Ollama](https://ollama.ai/) installed and running
- Completed Labs 1-4 (recommended)
- Basic familiarity with VS Code extensions

> **Note**: This lab is **local-only**. MCP requires a local server process and VS Code extension configuration, which cannot run on Google Colab.

## Setting up the environment

Ensure you are running Python 3.10 or later in a freshly created virtual environment.

In [ ]:
import sys
assert sys.version_info >= (3, 10) and sys.version_info < (3, 14), "Use Python 3.10, 3.11, 3.12, or 3.13 to run this notebook."

## Install Dependencies

In [ ]:
! echo "::group::Install Dependencies"
%pip install uv
! uv pip install docling-mcp mcp httpx
! echo "::endgroup::"

## Architecture Overview

This lab uses three components that communicate through MCP:

```text
+------------------+       MCP (stdio)       +------------------+
|                  | <---------------------> |                  |
|   Continue       |                         |   docling-mcp    |
|   (VS Code)      |                         |   server         |
|                  |                         |                  |
+--------+---------+                         +------------------+
         |                                          |
         | LLM calls                                | Docling
         v                                          v
+------------------+                         +------------------+
|   Granite 3.3    |                         |   Document       |
|   via Ollama     |                         |   Processing     |
+------------------+                         +------------------+
```

1. **Continue** (MCP Client): Open-source VS Code extension that acts as an AI coding assistant
2. **docling-mcp** (MCP Server): Exposes Docling tools via the MCP protocol
3. **Granite via Ollama** (LLM): The language model that decides when to use tools

When you ask Continue to process a document, Granite recognizes that it needs Docling's capabilities and invokes the appropriate MCP tools automatically.

## Understanding docling-mcp Tools

The [`docling-mcp`](https://github.com/docling-project/docling-mcp) package exposes three categories of tools:

### Conversion Tools

| Tool | Description |
| :--- | :--- |
| `is_document_in_local_cache` | Check if a document is already converted and cached |
| `convert_document_into_docling_document` | Convert a document (URL or local path) and store in cache |
| `convert_directory_files_into_docling_document` | Convert all files from a local directory |

### Document Generation Tools

| Tool | Description |
| :--- | :--- |
| `create_new_docling_document` | Create a new DoclingDocument from a prompt |
| `add_title_to_docling_document` | Add or update the title |
| `add_section_heading_to_docling_document` | Add a section heading |
| `add_paragraph_to_docling_document` | Add a paragraph |
| `open_list_in_docling_document` | Open a list structure |
| `add_list_items_to_list_in_docling_document` | Add items to an open list |
| `close_list_in_docling_document` | Close the current list |
| `add_table_in_html_format_to_docling_document` | Add an HTML-formatted table |
| `export_docling_document_to_markdown` | Export the document as Markdown |
| `save_docling_document` | Save to disk in markdown and JSON formats |

### Document Manipulation Tools

| Tool | Description |
| :--- | :--- |
| `page_thumbnail` | Generate a thumbnail image for a page |
| `get_overview_of_document_anchors` | Get a structured overview of a document |
| `search_for_text_in_document_anchors` | Search for text within document anchors |
| `get_text_of_document_item_at_anchor` | Retrieve text at a specific anchor |
| `update_text_of_document_item_at_anchor` | Update text at a specific anchor |
| `delete_document_items_at_anchors` | Delete items at specified anchors |

These tools allow an AI agent to **consume** existing documents, **create** new ones, and **manipulate** document content.

## Step 1: Set Up Ollama with Granite

First, ensure Ollama is installed and pull a Granite model that supports tool use.

### Install Ollama

Download from [ollama.ai](https://ollama.ai/) if not already installed.

### Pull a Granite Model

Open a terminal and run:

```bash
ollama pull granite3.3:8b
```

This pulls IBM Granite 3.3 (8B parameters), which supports tool calling — a requirement for MCP to work.

In [ ]:
import urllib.request
import urllib.error
import json

def check_ollama():
    """Check if Ollama is running and list available models."""
    try:
        req = urllib.request.Request("http://localhost:11434/api/tags", method="GET")
        with urllib.request.urlopen(req, timeout=5) as response:
            data = json.loads(response.read().decode())
            models = [m["name"] for m in data.get("models", [])]
            print("Ollama is running!")
            print(f"Available models: {models}")

            granite_models = [m for m in models if "granite" in m.lower()]
            if granite_models:
                print(f"\nGranite models found: {granite_models}")
            else:
                print("\nNo Granite models found. Please run:")
                print("  ollama pull granite3.3:8b")
            return True
    except (urllib.error.URLError, TimeoutError):
        print("Ollama is not running. Please start Ollama first.")
        print("  On macOS: Open the Ollama app")
        print("  On Linux: ollama serve")
        return False

ollama_available = check_ollama()

## Step 2: Verify docling-mcp Server

Let's verify that the docling-mcp server can be launched. The server will be started automatically by Continue (via stdio transport), but we can test it independently first.

### Quick Launch Test

In a separate terminal, run:

```bash
uvx --from docling-mcp docling-mcp-server --transport sse
```

This starts the server with SSE transport (for manual testing). You should see output indicating the server is ready. Press `Ctrl+C` to stop it when done.

> **Note**: When used with Continue, the server runs via `stdio` transport and is managed automatically — you do not need to start it manually.

In [ ]:
import subprocess
import shutil

# Check if uvx is available
uvx_path = shutil.which("uvx")
if uvx_path:
    print(f"uvx found at: {uvx_path}")
    # Verify docling-mcp can be resolved
    result = subprocess.run(
        ["uvx", "--from", "docling-mcp", "docling-mcp-server", "--help"],
        capture_output=True, text=True, timeout=120
    )
    if result.returncode == 0:
        print("docling-mcp server is available!")
        print("\nServer help output:")
        print(result.stdout[:500])
    else:
        print("Error launching docling-mcp:")
        print(result.stderr[:500])
else:
    print("uvx not found. Install it with: pip install uv")

## Step 3: Install and Configure Continue

### Install the Continue Extension

1. Open VS Code
2. Go to Extensions (`Ctrl+Shift+X` / `Cmd+Shift+X`)
3. Search for "Continue"
4. Install the **Continue** extension by [continue.dev](https://continue.dev)

### Configure Ollama as the Model Provider

After installation, Continue needs to be configured to use your local Granite model. Edit your Continue configuration file:

**Location**: `~/.continue/config.yaml`

Add or modify the models section:

```yaml
name: Docling Workshop
version: 0.0.1
schema: v1

models:
  - name: Granite 3.3
    provider: ollama
    model: granite3.3:8b
    apiBase: http://localhost:11434
    roles:
      - chat
      - edit
      - apply
    capabilities:
      - tool_use
```

### Configure the docling-mcp Server

Add the MCP server configuration to the same `config.yaml`:

```yaml
mcpServers:
  - name: Docling
    command: uvx
    args:
      - --from=docling-mcp
      - docling-mcp-server
```

### Complete config.yaml

Here is the full configuration file:

```yaml
name: Docling Workshop
version: 0.0.1
schema: v1

models:
  - name: Granite 3.3
    provider: ollama
    model: granite3.3:8b
    apiBase: http://localhost:11434
    roles:
      - chat
      - edit
      - apply
    capabilities:
      - tool_use

mcpServers:
  - name: Docling
    command: uvx
    args:
      - --from=docling-mcp
      - docling-mcp-server
```

### Verify in Continue

1. Open Continue in VS Code (click the Continue icon in the sidebar)
2. Switch to **Agent** mode (MCP tools are only available in Agent mode)
3. You should see the Granite model available in the model selector
4. The Docling MCP tools should appear in the tools panel

In [ ]:
from pathlib import Path

config_path = Path.home() / ".continue" / "config.yaml"
if config_path.exists():
    print(f"Continue config found at: {config_path}")
    print("\nCurrent contents:")
    print(config_path.read_text()[:1000])
else:
    print(f"Continue config not found at: {config_path}")
    print("This file will be created when you install and configure Continue.")
    print("See the instructions above for the recommended configuration.")

## Step 4: Using Docling Tools Through Continue

Now that everything is configured, let's use the Docling MCP tools through Continue.

> **Important**: Switch Continue to **Agent** mode. MCP tools are only available in Agent mode, not in Chat mode.

### Exercise 1: Convert a Document

In Continue's Agent mode, type:

```text
Convert the document at https://arxiv.org/pdf/2501.17887 and give me a summary
```

What happens behind the scenes:
1. Granite receives your request and recognizes it needs document conversion
2. It invokes the `convert_document_into_docling_document` MCP tool with the URL
3. docling-mcp converts the PDF using Docling
4. The structured content is returned to Granite
5. Granite summarizes the content for you

### Exercise 2: Create a New Document

Try asking Continue to create a document:

```text
Create a document titled "Meeting Notes" with a section "Action Items"
containing a bulleted list of: Review PR #42, Update documentation,
Schedule follow-up meeting. Then export it as markdown.
```

This will invoke multiple MCP tools in sequence:
1. `create_new_docling_document`
2. `add_title_to_docling_document`
3. `add_section_heading_to_docling_document`
4. `open_list_in_docling_document`
5. `add_list_items_to_list_in_docling_document`
6. `close_list_in_docling_document`
7. `export_docling_document_to_markdown`

### Exercise 3: Convert and Analyze

```text
Convert https://arxiv.org/pdf/2501.17887 and extract all the table data.
What are the key performance metrics reported?
```

## Step 5: Testing the MCP Server Programmatically (Optional)

While the primary workflow is through Continue, you can also test the docling-mcp server programmatically using the MCP Python SDK. This is useful for debugging or building custom integrations.

In [ ]:
try:
    from mcp import ClientSession, StdioServerParameters
    from mcp.client.stdio import stdio_client

    MCP_SDK_AVAILABLE = True
    print("MCP Python SDK is available")
except ImportError:
    MCP_SDK_AVAILABLE = False
    print("MCP Python SDK not installed. This is optional.")
    print("Install with: uv pip install mcp")

In [ ]:
if MCP_SDK_AVAILABLE:
    async def list_docling_tools():
        """Connect to docling-mcp and list available tools."""
        server_params = StdioServerParameters(
            command="uvx",
            args=["--from=docling-mcp", "docling-mcp-server"],
        )

        async with stdio_client(server_params) as (read, write):
            async with ClientSession(read, write) as session:
                await session.initialize()
                tools = await session.list_tools()

                print(f"docling-mcp exposes {len(tools.tools)} tools:\n")
                for tool in tools.tools:
                    print(f"  - {tool.name}")
                    if tool.description:
                        print(f"    {tool.description[:80]}")
                    print()

    await list_docling_tools()
else:
    print("Skipping -- MCP SDK not installed")

In [ ]:
if MCP_SDK_AVAILABLE:
    async def test_convert_document():
        """Test document conversion via MCP."""
        server_params = StdioServerParameters(
            command="uvx",
            args=["--from=docling-mcp", "docling-mcp-server"],
        )

        async with stdio_client(server_params) as (read, write):
            async with ClientSession(read, write) as session:
                await session.initialize()

                # Call the convert tool
                print("Converting document via MCP...")
                result = await session.call_tool(
                    "convert_document_into_docling_document",
                    arguments={"source": "https://arxiv.org/pdf/2501.17887"}
                )

                # Display a preview of the result
                content = str(result.content[0].text) if result.content else "No content"
                print(f"\nConversion result preview ({len(content)} chars):")
                print(content[:500])
                print("...")

    await test_convert_document()
else:
    print("Skipping -- MCP SDK not installed")

## MCP vs REST API: When to Use What

You've now seen two ways to expose Docling as a service:

| Feature | docling-serve (Lab 4) | docling-mcp (Lab 6) |
| :--- | :--- | :--- |
| **Protocol** | REST/HTTP | MCP (stdio/sse/http) |
| **Primary Consumer** | Your application code | AI agents/LLMs |
| **Discovery** | Read API docs manually | Tools auto-discovered by LLM |
| **Invocation** | You write HTTP requests | LLM decides when to call tools |
| **Use Case** | Microservices, batch processing | AI-assisted workflows, copilots |
| **Scaling** | Load balancers, containers | Per-agent instance |
| **Best For** | Production pipelines | Interactive AI experiences |

### The Key Difference

With **docling-serve**, you (the developer) decide when to call the API. With **docling-mcp**, the AI agent decides when to invoke Docling tools based on the user's natural language request.

## Troubleshooting

### Ollama Issues

| Problem | Solution |
| :--- | :--- |
| "Ollama is not running" | Start the Ollama app (macOS) or run `ollama serve` (Linux) |
| Model not found | Run `ollama pull granite3.3:8b` |
| Slow responses | Try a smaller model: `ollama pull granite3.3:2b` |

### Continue Issues

| Problem | Solution |
| :--- | :--- |
| MCP tools not showing | Ensure you are in **Agent** mode, not Chat mode |
| "Cannot connect to MCP server" | Verify `uvx` is on your PATH: `which uvx` |
| Model not appearing | Check `config.yaml` syntax and restart VS Code |
| Tool calls failing | Check Continue output panel for error logs |

### docling-mcp Issues

| Problem | Solution |
| :--- | :--- |
| "uvx not found" | Install uv: `pip install uv` or `brew install uv` |
| Import errors | Run `uvx --from docling-mcp docling-mcp-server --help` to debug |
| Conversion timeout | Large documents may take time; try a smaller PDF first |

## Summary

In this lab, you learned how to:

1. **Set up the MCP stack**: Ollama (Granite) + Continue + docling-mcp
2. **Configure MCP tools**: Connect docling-mcp to Continue via config.yaml
3. **Use Docling through AI agents**: Convert and create documents via natural language
4. **Compare approaches**: When to use REST API (Lab 4) vs MCP (Lab 6)

### The AI Agent Paradigm

MCP represents a shift in how we integrate tools with AI:
- **Before MCP**: Developers write code to call APIs
- **With MCP**: AI agents discover and invoke tools autonomously

Docling's MCP integration means document processing becomes a natural capability of any AI agent, accessible through conversation rather than code.

### Resources

- [docling-mcp GitHub](https://github.com/docling-project/docling-mcp)
- [Model Context Protocol Specification](https://modelcontextprotocol.io/)
- [Continue Documentation](https://docs.continue.dev/)
- [Ollama](https://ollama.ai/)
- [Docling Documentation](https://docling-project.github.io/docling/)
- [IBM Granite Models](https://www.ibm.com/granite)